# ex02 · 填充与步幅（对应教材 6.3）

> **做题流程**：先套公式预测尺寸，再从零实现带填充/步幅的卷积。
> **做完再看** `solutions/ex02-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 输出尺寸公式：**⌊(n − k + 2p) / s⌋ + 1**。

## 题 1 🌱 预测输出形状（简短）

对 8×8 输入，套公式算下面几个卷积的输出形状，再运行对照（不写代码，只预测）。

In [4]:
import torch
from torch import nn

def comp_conv2d(conv2d, X):
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])

X = torch.rand(size=(8, 8))
s1 = list(comp_conv2d(nn.Conv2d(1, 1, kernel_size=3, padding=1), X).shape)
s2 = list(comp_conv2d(nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2), X).shape)
s3 = list(comp_conv2d(nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4)), X).shape)
print('s1 =', s1, ' s2 =', s2, ' s3 =', s3)

s1 = [8, 8]  s2 = [4, 4]  s3 = [2, 2]


## 题 2 🔧 从零实现带填充和步幅的卷积（TODO 6.3）

上面的 nn.Conv2d 一步就做完了。现在**从零**实现：先手动补零，再按步幅滑动窗口，复用 ex01 的 corr2d 做乘加。补全 conv2d_pad。

In [9]:
def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

def conv2d_pad(X, K, padding=0, stride=1):
    # TODO 6.3: 先手动补零（padding 圈 0），再按 stride 步进取窗口做互相关
    # 提示: 补零 = 造一个 (H+2p, W+2p) 的全 0 矩阵，把 X 填进中间
    h, w = K.shape
    H, W = X.shape
    padtensor = torch.zeros(H + 2*padding, W + 2*padding)
    for i in range(H):
        for j in range(W):
            padtensor[i + 1][j + 1] = X[i][j]
    padcorr2d = corr2d(padtensor, K)
    return padcorr2d[::stride , ::stride]    #先跳和后跳是一样的

In [10]:
try:
    X = torch.rand(size=(8, 8))
    K = torch.rand(size=(3, 3))
    # 与 nn.Conv2d 对比：padding=1, stride=1
    conv = nn.Conv2d(1, 1, kernel_size=3, padding=1)
    with torch.no_grad():
        conv.weight.copy_(K.reshape(1, 1, 3, 3)); conv.bias.zero_()
    mine = conv2d_pad(X, K, padding=1)
    ref = comp_conv2d(conv, X)
    assert torch.allclose(mine, ref, atol=1e-6), f'{mine}\n{ref}'
    # 再加 stride
    conv2 = nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)
    with torch.no_grad():
        conv2.weight.copy_(K.reshape(1, 1, 3, 3)); conv2.bias.zero_()
    mine2 = conv2d_pad(X, K, padding=1, stride=2)
    ref2 = comp_conv2d(conv2, X)
    assert torch.allclose(mine2, ref2, atol=1e-6)
    print('✓ conv2d_pad 与 nn.Conv2d 结果一致')
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

✓ conv2d_pad 与 nn.Conv2d 结果一致


## 小结与面试衔接

- 填充 p：控制输出尺寸（padding=k//2 实现「same」卷积）
- 步幅 s：下采样、减小特征图
- 从零实现一遍，你就清楚了 nn.Conv2d 内部到底做了什么；尺寸公式是面试必考